# Noonchi Translator — mBART Fine-tuning

Fine-tunes `facebook/mbart-large-50-many-to-many-mmt` on the formality-labeled EN–KR corpus.

**Before running:** Set Runtime → Change runtime type → GPU (T4 or better).

**Training time estimates:**
- 50K rows, 3 epochs, batch 4 × grad_accum 8 on T4: ~5–8 hours
- Full 423K rows: use A100 (Colab Pro) or a university cluster

All checkpoints are saved to Google Drive so a session disconnect doesn't lose progress.

In [1]:
# Cell 1 — Verify GPU is assigned before doing anything else
!nvidia-smi

Fri May  1 16:52:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Cell 2 — Mount Google Drive
# All model checkpoints write here so disconnects don't lose progress.
from google.colab import drive
drive.mount('/content/drive')

import os

Mounted at /content/drive


In [3]:
# Cell 3 — Install dependencies
# transformers 4.41+ required for eval_strategy rename and text_target tokenizer API.
# accelerate is a hard requirement of HuggingFace Trainer (often missing from requirements).
# mecab-ko (via konlpy install script) is needed for formality_accuracy evaluation.
!pip install -q 'transformers>=4.41.0' 'accelerate>=0.26.0' datasets sentencepiece sacrebleu
!pip install -q mecab-python3 konlpy
!bash <(curl -s https://raw.githubusercontent.com/konlpy/konlpy/master/scripts/mecab.sh)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 591.4/591.4 kB 15.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 98.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.0/438.0 kB 43.4 MB/s eta 0:00:00
Install mecab-ko
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 1381k  100 1381k    0     0  1133k      0  0:00:01  0:00:01 --:--:-- 1927k
mecab-0.996-ko-0.9.2/
mecab-0.996-ko-0.9.2/example/
mecab-0.996-ko-0.9.2/example/example.cpp
mecab-0.996-ko-0.9.2/example/example_lattice.cpp
mecab-0.996-ko-0.9.2/example/example_lattice.c
mecab-0.996-ko-0.9.2/example/example.c
mecab-0.996-ko-0.9.2/example/thread_test.cpp
mecab-0.996-ko-0.9.2/mecab-config.in
mecab-0.996-ko-0.9.2

In [ ]:
# Cell 4 — Clone repo and set up Python path
# Add your GitHub token to Colab Secrets (key icon in sidebar) as GITHUB_TOKEN
from google.colab import userdata
token = 'GITHUB_TOKEN'

import os, subprocess, sys, importlib

repo_path = '/content/noonchi-translator'
if os.path.exists(repo_path):
    subprocess.run(['git', '-C', repo_path, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone',
        f'https://{token}@github.com/cyatco01/noonchi-translator.git', repo_path],
        check=True)

sys.path.insert(0, repo_path)
importlib.invalidate_caches()
%cd /content/noonchi-translator

from backend.model.train import load_model_and_tokenizer
from backend.model.dataset import load_split
print('Imports OK')


/content/noonchi-translator
Imports OK


In [5]:
# Cell 5 — Copy training data from Drive
# Upload train.tsv and val.tsv to /content/drive/MyDrive/noonchi/ first.
# These files are too large to commit to git (~50MB each).
!cp /content/drive/MyDrive/noonchi/train.tsv data/train.tsv
!cp /content/drive/MyDrive/noonchi/val.tsv data/val.tsv
!cp /content/drive/MyDrive/noonchi/test.tsv data/test.tsv

# Verify line counts
!wc -l data/train.tsv data/val.tsv data/test.tsv

  338959 data/train.tsv
   42370 data/val.tsv
   42373 data/test.tsv
  423702 total


In [ ]:
# # Cell 6 — Smoke test: load model and one batch before committing to full training
# # If this fails, debug here rather than 30 minutes into a training run.
model, tokenizer = load_model_and_tokenizer()
ds = load_split('data/val.tsv', tokenizer)
sample = ds[0]
print('Dataset size:', len(ds))
print('Sample keys:', list(sample.keys()))
print('input_ids length:', len(sample['input_ids']))
print('labels length:', len(sample['labels']))
print('decoder_start_token_id:', model.config.decoder_start_token_id)
print('forced_bos_token_id:', model.config.forced_bos_token_id)
del model  # free VRAM before training

In [6]:
# Cell 7 — Train
# --max-rows 20000: quick sanity-check run (~2h on T4) — confirms training is healthy before scaling up
# Change to --max-rows 50000 once you've verified real Korean output from Cell 9
# Remove --max-rows entirely to train on the full 423K dataset (needs A100 or ~40+ hours)
# Add --resume to pick up from the latest checkpoint if the session crashed
!python -m backend.model.train --data data/train.tsv --output /content/drive/MyDrive/noonchi/checkpoints/noonchi-mbart --max-rows 50000


INFO: HTTP Request: HEAD https://huggingface.co/facebook/mbart-large-50-many-to-many-mmt/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/mbart-large-50-many-to-many-mmt/e30b6cb8eb0d43a0b73cab73c7676b9863223a30/tokenizer_config.json "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://huggingface.co/api/resolve-cache/models/facebook/mbart-large-50-many-to-many-mmt/e30b6cb8eb0d43a0b73cab73c7676b9863223a30/tokenizer_config.json "HTTP/1.1 200 OK"
tokenizer_config.json: 100% 529/529 [00:00<00:00, 2.02MB/s]
INFO: HTTP Request: GET https://huggingface.co/api/models/facebook/mbart-large-50-many-to-many-mmt/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO: HTTP Request: GET https://huggingface.co/api/models/facebook/mbart-large-50-many-to-many-mmt/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/f

In [ ]:
# #if runtime disconnects
# !cd /content/noonchi-translator && git pull                                     
# import sys                                                                      
# sys.path.insert(0, '/content/noonchi-translator')                               
# %cd /content/noonchi-translator    

In [ ]:
# !bash <(curl -s https://raw.githubusercontent.com/konlpy/konlpy/master/scripts/mecab.sh)   

In [ ]:
# if run time disconnected or edits made to git
!git -C /content/noonchi-translator pull     

In [7]:
# Cell 8 — Evaluate on held-out test split
# Sanity bars (20K model): chrF > 10, formality_accuracy > 0.50
# Sanity bars (50K model): chrF > 20, formality_accuracy > 0.60
# Target bars (full model): chrF > 30, formality_accuracy > 0.80
#
# ⚠️  If you just retrained: delete the old cache first, or Cell 8 will reuse stale outputs.
#     ! rm -f /content/drive/MyDrive/noonchi/hypotheses_cache.txt
!python -m backend.model.evaluate --model /content/drive/MyDrive/noonchi/checkpoints/noonchi-mbart --test data/test.tsv --batch-size 8 --num-beams 4 --hypotheses-cache /content/drive/MyDrive/noonchi/hypotheses_cache.txt --output /content/drive/MyDrive/noonchi/eval_results.json


Loading weights: 100% 516/516 [00:00<00:00, 862.42it/s, Materializing param=model.shared.weight]                                   
INFO: Evaluating on 42,372 test rows from data/test.tsv
Translating: 100% 5297/5297 [1:00:56<00:00,  1.45batch/s]
INFO: Hypotheses cached to /content/drive/MyDrive/noonchi/hypotheses_cache.txt — metrics can be re-run without re-translating
INFO: Computing metrics...
Formality accuracy: 100% 42372/42372 [00:02<00:00, 18331.76sent/s]

=== Evaluation Results ===
  chrF: 28.6394
  BLEU: 12.8139
  formality_accuracy: 0.8512

INFO: Evaluation complete.
Results saved to /content/drive/MyDrive/noonchi/eval_results.json


In [8]:
# Cell 9 — Inspect sample predictions
# Run after Cell 8 to eyeball what the model actually generated.
# If outputs are 요요요요요 or empty, decoder alignment is still broken.
# ⚠️  Delete the cache and retrain before running Cell 8 again if you see garbage here.
cache_path = "/content/drive/MyDrive/noonchi/hypotheses_cache.txt"
test_path = "data/test.tsv"

import csv
hypotheses = open(cache_path, encoding="utf-8").read().splitlines()

references, en_inputs = [], []
with open(test_path, encoding="utf-8", newline="") as f:
    reader = csv.reader(f, delimiter="\t")
    next(reader)
    for row in reader:
        if len(row) == 3:
            en_inputs.append(row[0])
            references.append(row[1])

print(f"Total hypotheses: {len(hypotheses)}, references: {len(references)}\n")
for i in range(min(10, len(hypotheses))):
    print(f"[{i}] EN:  {en_inputs[i]}")
    print(f"     REF: {references[i]}")
    print(f"     HYP: {hypotheses[i]}")
    print()


Total hypotheses: 42372, references: 42372

[0] EN:  You're a disappointment,young lady,a big,big,disappointment.
     REF: 정말 실망이구나, 다 큰 아가씨가! 정말이지 크게 실망했어.
     HYP: 넌 실망이야, 젊어, 큰, 큰 실망

[1] EN:  You did promise me after Julia died that we would all be reconciled.
     REF: 줄리아가 죽은 후에 우리 모두 화해할꺼라고 당신은 나와 약속했잖아요.
     HYP: 줄리아가 죽고 난 후에 우리 모두 화해된다고 약속하셨잖아요

[2] EN:  And listen, I know Drew is a kid whith a lot of energy, but we just put a bunch of screws in your back, so you gotta take it easy for a few weeks.
     REF: 다음 주 검진 때 봐요 그리고 드류가 활발한 아이라는 건 알지만 우리가 방금 환자분 등에 나사를 한 뭉치나 넣었잖습니다
     HYP: 닥터 드류는 힘이 많은 아이라는 건 알지만 우린 방금 당신 뒤에 스위치를 끼어 놨습니다

[3] EN:  We'll go ashore, meet up with Packard.
     REF: 육지로 올라가서 패커드를 만나죠
     HYP: 해변에 가서 파커드와 만나요

[4] EN:  I'm lost. Please, can you help me?
     REF: 길을 잃었어요 제발 도와주실래요?
     HYP: 잃어버렸어요 도와주실래요?

[5] EN:  The new software update includes several important security improvements.
     REF: 새로운 소프트웨어 업데이트에는 여러 중요한 보안 개선 사항이 포함되어 있습니다.
     HY

In [9]:
# Cell 10 — Conditioning sanity check
# Runs 3 sentences through all 3 formality tokens.
# Outputs should have visibly different endings (습니다 / 아요-어요 / 아-어).
# If all 3 are identical, the model is ignoring the conditioning token.
import torch
from backend.model.train import load_model_and_tokenizer, TGT_LANG

model, tokenizer = load_model_and_tokenizer('/content/drive/MyDrive/noonchi/checkpoints/noonchi-mbart')
model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
ko_id = tokenizer.lang_code_to_id[TGT_LANG]

test_sentences = [
    "Can you help me with this?",
    "The meeting has been rescheduled.",
    "I want to eat dinner with you.",
]

for en in test_sentences:
    print(f"\nEN: {en}")
    for formality in ["formal", "polite", "casual"]:
        inputs = tokenizer(f"<{formality}> {en}", return_tensors="pt").to(device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                forced_bos_token_id=ko_id,
                num_beams=4,
                max_new_tokens=80,
                no_repeat_ngram_size=3,
                repetition_penalty=1.2,
            )
        print(f"  <{formality}>  {tokenizer.decode(out[0], skip_special_tokens=True)}")


Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]


EN: Can you help me with this?
  <formal>  이걸 도와주실 수 있습니까?
  <polite>  이걸 도와주실래요?
  <casual>  이거 도와줄 수 있어?

EN: The meeting has been rescheduled.
  <formal>  회의가 재일정되었습니다.
  <polite>  회의 일정이 변경됐어요
  <casual>  회의 일정이 변경됐어

EN: I want to eat dinner with you.
  <formal>  당신과 함께 저녁을 먹고 싶습니다
  <polite>  당신과 함께 저녁을 먹고 싶어요
  <casual>  당신과 함께 저녁을 먹고 싶어
